# Scout Delta Lake Quickstart

## Setup

First setup your Trino connection, this allows you to query the Delta Lake tables through pandas using SQL.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    f"trino://{os.environ['TRINO_USER']}@{os.environ['TRINO_HOST']}:{os.environ['TRINO_PORT']}/"
    f"{os.environ['TRINO_CATALOG']}/{os.environ['TRINO_SCHEMA']}?http_scheme={os.environ['TRINO_SCHEME']}"
)

In [ ]:
# We will filter for our IRB-approved facilities in when querying
# Adjust as needed.
FACILITIES = ['BJH', 'WUSM', 'BJCMG', 'SLCH']
FACILITIES_SQL = "'" + "', '".join(FACILITIES) + "'"

## Exploring the data

In [ ]:
default_facility_count = pd.read_sql(f"""
    SELECT sending_facility, COUNT(*) AS count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY sending_facility
    ORDER BY count DESC
""", engine)
default_facility_count

In [ ]:
modality_count = pd.read_sql(f"""
    SELECT modality, COUNT(*) AS count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY modality
    ORDER BY count DESC
""", engine)
modality_count

In [ ]:
sex_distribution = pd.read_sql(f"""
    SELECT sex, modality, COUNT(*) AS count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality IN ('CT', 'MR')
    GROUP BY sex, modality
    ORDER BY modality, sex, count DESC
""", engine)
sex_distribution

## Querying reports

Parsed report sections (`report_section_findings`, `report_section_impression`, `report_section_addendum`, `report_section_technician_note`) and the full `report_text` are available for text-based filtering. You can also filter by structured fields like diagnosis codes, modality, or date.

In [ ]:
# Print available columns
columns_df = pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'default' AND table_name = 'reports_latest'
    ORDER BY ordinal_position
""", engine)
columns_df

In [ ]:
# Search for "pulmonary embolism" in the report impression section or any I26% diagnosis code, excluding negations in the impression text.
pe_cohort = pd.read_sql(f"""
    SELECT accession_number, modality, service_name, requested_dt,
           report_section_impression, diagnoses
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND year = 2024
      AND (
          any_match(diagnoses, d -> d.diagnosis_code LIKE 'I26%')
          OR (
              regexp_like(report_section_impression, '(?i)pulmonary embolism')
              AND NOT regexp_like(report_section_impression,
                  '(?i)(?:no|without|negative for|absence of|ruled? out)[^.;:]{{0,40}}pulmonary embolism')
          )
      )
    LIMIT 100
""", engine)
pe_cohort

In [ ]:
# Search for head/brain MRs in adults in 2024 with a free-text filter on the service name.
combined_search = pd.read_sql(f"""
    SELECT accession_number, patient_age, sex, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality = 'MR'
      AND regexp_like(service_name, '(?i)head|brain')
      AND patient_age >= 18
      AND year = 2024
    LIMIT 100
""", engine)
combined_search

## Longitudinal patient queries

Various use cases require a user be able to identify a particular patient’s reports over time in the delta lake. This need is complicated by the fact that the form of patient IDs available in the underlying reports has changed twice, so a given patient may have reports with three distinct sets of IDs. The `reports_latest_epic_view` view bridges these via three columns on top of the base report fields:

- **`scout_patient_id`** — Scout's internal patient identifier, always populated. Use as the `GROUP BY` key for per-patient aggregates.
- **`resolved_epic_mrn`** — bridged Epic MRN. Filter on this to look up a patient by Epic MRN.
- **`resolved_mpi`** — bridged legacy MPI.

In [ ]:
# Longitudinal brain MRI cohort — patients with brain/head MRIs spanning 5+ years.
longitudinal_brain = pd.read_sql(f"""
    SELECT ANY_VALUE(resolved_epic_mrn) AS epic_mrn,
           ANY_VALUE(resolved_mpi)      AS mpi,
           COUNT(*) AS brain_mri_count,
           MIN(requested_dt) AS earliest,
           MAX(requested_dt) AS latest,
           date_diff('year', MIN(requested_dt), MAX(requested_dt)) AS years_span
    FROM reports_latest_epic_view
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality = 'MR'
      AND regexp_like(service_name, '(?i)brain|head')
    GROUP BY scout_patient_id
    HAVING COUNT(*) >= 2
       AND date_diff('year', MIN(requested_dt), MAX(requested_dt)) >= 5
    ORDER BY years_span DESC, brain_mri_count DESC
    LIMIT 10
""", engine)
longitudinal_brain

In [ ]:
# Drill into a specific patient's report history
patient_mrn = longitudinal_brain.iloc[0]['epic_mrn']

patient_history = pd.read_sql(f"""
    SELECT requested_dt, modality, service_name, accession_number,
           report_section_impression
    FROM reports_latest_epic_view
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND resolved_epic_mrn = '{patient_mrn}'
    ORDER BY requested_dt
""", engine)
patient_history

In [ ]:
# Lung nodule surveillance — patients with any 3 chest CTs within an 18-month rolling window and at least one impression mentioning a nodule. 
lung_surveillance = pd.read_sql(f"""
    WITH chest_cts AS (
        SELECT scout_patient_id, requested_dt, report_section_impression,
               resolved_epic_mrn, resolved_mpi
        FROM reports_latest_epic_view
        WHERE sending_facility IN ({FACILITIES_SQL})
          AND modality = 'CT'
          AND regexp_like(service_name, '(?i)chest|thorax|lung')
    ),
    surveillance_bursts AS (
        SELECT scout_patient_id,
               MIN(requested_dt) AS burst_first_ct,
               MIN_BY(third_ct_dt, requested_dt) AS burst_third_ct
        FROM (
            SELECT scout_patient_id, requested_dt,
                   LEAD(requested_dt, 2) OVER (PARTITION BY scout_patient_id ORDER BY requested_dt) AS third_ct_dt
            FROM chest_cts
        )
        WHERE date_diff('month', requested_dt, third_ct_dt) <= 18
        GROUP BY scout_patient_id
    ),
    nodule_patients AS (
        SELECT DISTINCT scout_patient_id
        FROM chest_cts
        WHERE regexp_like(report_section_impression, '(?i)nodule')
    )
    SELECT ANY_VALUE(c.resolved_epic_mrn) AS epic_mrn,
           ANY_VALUE(c.resolved_mpi)      AS mpi,
           COUNT(*) AS total_chest_cts,
           COUNT(*) FILTER (WHERE regexp_like(c.report_section_impression, '(?i)nodule')) AS nodule_mentions,
           ANY_VALUE(s.burst_first_ct)  AS burst_first_ct,
           ANY_VALUE(s.burst_third_ct)  AS burst_third_ct
    FROM chest_cts c
    JOIN surveillance_bursts s ON c.scout_patient_id = s.scout_patient_id
    JOIN nodule_patients n ON c.scout_patient_id = n.scout_patient_id
    GROUP BY c.scout_patient_id
    ORDER BY nodule_mentions DESC, total_chest_cts DESC
    LIMIT 10
""", engine)
lung_surveillance

In [ ]:
# Ischemic stroke patients from 2024 with all their prior imaging summarized.
stroke_history = pd.read_sql(f"""
    WITH stroke_patients AS (
        SELECT DISTINCT scout_patient_id
        FROM reports_latest_epic_view
        WHERE sending_facility IN ({FACILITIES_SQL})
          AND year = 2024
          AND any_match(diagnoses, d -> d.diagnosis_code LIKE 'I63%')
    )
    SELECT ANY_VALUE(r.resolved_epic_mrn) AS epic_mrn,
           ANY_VALUE(r.resolved_mpi)      AS mpi,
           COUNT(*) AS prior_reports,
           MIN(r.requested_dt) AS earliest_imaging,
           MAX(r.requested_dt) AS latest_imaging,
           array_agg(DISTINCT r.modality) AS modalities
    FROM reports_latest_epic_view r
    JOIN stroke_patients sp ON r.scout_patient_id = sp.scout_patient_id
    WHERE r.sending_facility IN ({FACILITIES_SQL})
      AND r.requested_dt < DATE '2024-01-01'
    GROUP BY r.scout_patient_id
    ORDER BY prior_reports DESC
    LIMIT 10
""", engine)
stroke_history

## Bulk patient lookup from a CSV

Upload a CSV of Epic MRNs to `/home/jovyan/Scout/patient_ids.csv` and pull all their reports via the `reports_latest_epic_view` view and the `resolved_epic_mrn` column.

In [ ]:
patient_ids_path = '/home/jovyan/Scout/patient_ids.csv'
if not os.path.exists(patient_ids_path):
    pd.DataFrame({'epic_mrn': ['REPLACE_WITH_REAL_MRN']}).to_csv(patient_ids_path, index=False)
    print(f"Created sample CSV at {patient_ids_path} — edit with real MRNs")

patient_ids_df = pd.read_csv(patient_ids_path, dtype=str)
mrns = patient_ids_df['epic_mrn'].dropna().str.strip().tolist()
print(f"Loaded {len(mrns)} patient IDs from CSV")

In [ ]:
mrn_list = "', '".join(mrns)
search = pd.read_sql(f"""
    SELECT accession_number, version_id, epic_mrn, resolved_epic_mrn,
           modality, service_name, requested_dt, report_text
    FROM reports_latest_epic_view
    WHERE resolved_epic_mrn IN ('{mrn_list}')
      AND sending_facility IN ({FACILITIES_SQL})
    ORDER BY resolved_epic_mrn, requested_dt
    LIMIT 100
""", engine)
search.head()

In [ ]:
# Write results to CSV.
search.to_csv('/home/jovyan/Scout/patient_list_results.csv', index=False)

## Installing Additional Packages

The base Jupyter environment includes Trino client, pandas, matplotlib, seaborn, scikit-learn, statsmodels, pyarrow, and other core data analysis packages. For ML, NLP, or other specialized work, create a new conda environment:

```bash
# Create an environment with specific packages
mamba create -n my-env python=3.11 ipykernel pytorch transformers scikit-learn -y

# Or use the sample environment file (in ~/Scout/environment.yml)
mamba env create -f ~/Scout/environment.yml
```

**Important:** Include `ipykernel` in every environment you create. The `nb_conda_kernels` extension uses it to discover the environment as a Jupyter kernel. Without it, your environment won't appear in the kernel list.

After creating an environment, refresh the launcher to see it as a selectable kernel.

Environments are stored on your persistent home directory (`/home/jovyan/.conda/envs/`) and survive server restarts. In air-gapped deployments, package requests are routed through a proxy transparently — no extra configuration needed.

For more details, see the [Tips & Best Practices](https://washu-scout.readthedocs.io/en/latest/tips.html#installing-additional-packages) documentation.